In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

df = pd.read_csv("../data/raw/customs_kaggle_raw.csv")
print(df.shape)

(14873, 22)


In [2]:
# dropping columns that are just IDs, not real features
df = df.drop(columns=["Declaration_ID", "Importer_BIN", "Agent_ID"])

# engineered features confirmed useful in EDA
df["weight_diff"] = df["Scanned_Weight_KG"] - df["Declared_Weight_KG"]
df["value_per_kg"] = df["Declared_Value_BDT"] / df["Declared_Weight_KG"]

# extracting month from date, might catch seasonal fraud patterns
df["Date"] = pd.to_datetime(df["Date"])
df["month"] = df["Date"].dt.month
df = df.drop(columns=["Date"])

print(df.shape)

(14873, 21)


In [3]:
# these are just placeholder fills, not statistical, so safe pre-split
df["Declared_Description"] = df["Declared_Description"].fillna("")
df["Country_of_Origin"] = df["Country_of_Origin"].fillna("Unknown")
df["Country_of_Departure"] = df["Country_of_Departure"].fillna("Unknown")
df["Statistical_Unit"] = df["Statistical_Unit"].fillna("Unknown")

print(df.isna().sum()[df.isna().sum() > 0])

Scanned_Weight_KG    1527
SD_Rate               279
weight_diff          1527
dtype: int64


In [4]:
y = df["Fraud"]
X = df.drop(columns=["Fraud"])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)

(11898, 20) (2975, 20)


In [5]:
# scanned weight and SD_Rate still have missing numeric values
num_missing_cols = ["Scanned_Weight_KG", "SD_Rate"]

for col in num_missing_cols:
    median_val = X_train[col].median()
    X_train[col] = X_train[col].fillna(median_val)
    X_test[col] = X_test[col].fillna(median_val)

# recomputing weight_diff since Scanned_Weight_KG was just filled
X_train["weight_diff"] = X_train["Scanned_Weight_KG"] - X_train["Declared_Weight_KG"]
X_test["weight_diff"] = X_test["Scanned_Weight_KG"] - X_test["Declared_Weight_KG"]

print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

0 0


In [6]:
# too many unique values for one-hot: HS code, countries, statistical unit
freq_cols = ["Declared_HS_Code", "Country_of_Origin", "Country_of_Departure", "Statistical_Unit"]
freq_maps = {}

for col in freq_cols:
    freq_map = X_train[col].value_counts()
    freq_maps[col] = freq_map
    X_train[col + "_freq"] = X_train[col].map(freq_map)
    X_test[col + "_freq"] = X_test[col].map(freq_map).fillna(0)

X_train = X_train.drop(columns=freq_cols)
X_test = X_test.drop(columns=freq_cols)

print(X_train.shape, X_test.shape)

(11898, 20) (2975, 20)


In [7]:
X_train = pd.get_dummies(X_train, columns=["Port_of_Entry"], prefix="port")
X_test = pd.get_dummies(X_test, columns=["Port_of_Entry"], prefix="port")

# align test columns to train columns (in case a port only appears in one split)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

print(X_train.shape, X_test.shape)

(11898, 28) (2975, 28)


In [8]:
# label encode, but store separately so we can build Variant A / Variant B
risk_map = {"Green": 0, "Yellow": 1, "Red": 2}

risk_train = X_train["Risk_Channel"].map(risk_map)
risk_test = X_test["Risk_Channel"].map(risk_map)

X_train = X_train.drop(columns=["Risk_Channel"])
X_test = X_test.drop(columns=["Risk_Channel"])

print(risk_train.value_counts())

Risk_Channel
0    7587
1    2950
2    1361
Name: count, dtype: int64


In [9]:
text_col = "Declared_Description"
numeric_cols = [c for c in X_train.columns if c != text_col and X_train[c].dtype != "bool"]

scaler = StandardScaler()
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

print(X_train[numeric_cols].describe().loc[["mean", "std"]])

      Declared_Weight_KG  Declared_Value_BDT  Scanned_Weight_KG  \
mean       -6.091390e-17       -7.763536e-18      -5.971951e-17   
std         1.000042e+00        1.000042e+00       1.000042e+00   

      Exchange_Rate       CD_Rate       SD_Rate      VAT_Rate      AIT_Rate  \
mean  -5.162154e-15  1.612427e-16  3.702610e-17 -2.191706e-16  4.998523e-16   
std    1.000042e+00  1.000042e+00  1.000042e+00  1.000042e+00  1.000042e+00   

           RD_Rate      TTI_Rate   weight_diff  value_per_kg         month  \
mean -3.120344e-17  2.472388e-16  7.166341e-18  1.463128e-17  4.120646e-17   
std   1.000042e+00  1.000042e+00  1.000042e+00  1.000042e+00  1.000042e+00   

      Declared_HS_Code_freq  Country_of_Origin_freq  \
mean          -9.555122e-18            1.122727e-16   
std            1.000042e+00            1.000042e+00   

      Country_of_Departure_freq  Statistical_Unit_freq  
mean              -1.033148e-16           1.507918e-16  
std                1.000042e+00           1.0

In [10]:
tfidf = TfidfVectorizer(max_features=100)

tfidf_train = tfidf.fit_transform(X_train[text_col]).toarray()
tfidf_test = tfidf.transform(X_test[text_col]).toarray()

tfidf_train_df = pd.DataFrame(tfidf_train, columns=[f"tfidf_{i}" for i in range(tfidf_train.shape[1])], index=X_train.index)
tfidf_test_df = pd.DataFrame(tfidf_test, columns=[f"tfidf_{i}" for i in range(tfidf_test.shape[1])], index=X_test.index)

print(tfidf_train_df.shape)

(11898, 100)


In [11]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

emb_train = embed_model.encode(X_train[text_col].tolist(), show_progress_bar=False)
emb_test = embed_model.encode(X_test[text_col].tolist(), show_progress_bar=False)

emb_train_df = pd.DataFrame(emb_train, columns=[f"emb_{i}" for i in range(emb_train.shape[1])], index=X_train.index)
emb_test_df = pd.DataFrame(emb_test, columns=[f"emb_{i}" for i in range(emb_test.shape[1])], index=X_test.index)

print(emb_train_df.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

(11898, 384)


In [12]:
base_train = X_train.drop(columns=[text_col])
base_test = X_test.drop(columns=[text_col])

# Variant A: no Risk_Channel (deployable model)
A_train_tfidf = pd.concat([base_train, tfidf_train_df], axis=1)
A_test_tfidf = pd.concat([base_test, tfidf_test_df], axis=1)

A_train_emb = pd.concat([base_train, emb_train_df], axis=1)
A_test_emb = pd.concat([base_test, emb_test_df], axis=1)

# Variant B: with Risk_Channel (upper-bound sanity check)
B_train_tfidf = A_train_tfidf.copy()
B_train_tfidf["risk_channel"] = risk_train
B_test_tfidf = A_test_tfidf.copy()
B_test_tfidf["risk_channel"] = risk_test

B_train_emb = A_train_emb.copy()
B_train_emb["risk_channel"] = risk_train
B_test_emb = A_test_emb.copy()
B_test_emb["risk_channel"] = risk_test

# save all variants plus targets
data_to_save = {
    "A_train_tfidf": A_train_tfidf, "A_test_tfidf": A_test_tfidf,
    "A_train_emb": A_train_emb, "A_test_emb": A_test_emb,
    "B_train_tfidf": B_train_tfidf, "B_test_tfidf": B_test_tfidf,
    "B_train_emb": B_train_emb, "B_test_emb": B_test_emb,
    "y_train": y_train, "y_test": y_test
}

with open("../data/processed/processed_data.pkl", "wb") as f:
    pickle.dump(data_to_save, f)

print("saved. shapes:")
for k, v in data_to_save.items():
    print(k, v.shape)

saved. shapes:
A_train_tfidf (11898, 126)
A_test_tfidf (2975, 126)
A_train_emb (11898, 410)
A_test_emb (2975, 410)
B_train_tfidf (11898, 127)
B_test_tfidf (2975, 127)
B_train_emb (11898, 411)
B_test_emb (2975, 411)
y_train (11898,)
y_test (2975,)
